# Miniproyecto 3 · Encontrar grupos: k-means, jerárquico y mezcla de gaussianas
*Curso de Machine Learning · Miniproyecto · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

El *clustering* agrupa datos **sin etiquetas**: segmentar clientes, comprimir imágenes, detectar objetos raros… (Jain,
2010). Primero en 2D, donde se ve todo, y después con estrellas variables, donde no.

**Entrega:** tu copia en Drive, con todas las celdas ejecutadas y comentadas (100 puntos). No necesita GPU.

In [ ]:
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Elige los datos ──────────────────────────────────────────────────────────────────────────────
# "abiertos":  datos públicos o generados aquí (funciona siempre).
# "diplomado": los datos originales del miniproyecto del diplomado; sube los CSV a tu Drive, en
#              MyDrive/MiAtlas-ML/datos-diplomado/ (solo si tienes esos archivos).
DATOS = "abiertos"
if DATOS == "diplomado":
    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        CARPETA_DATOS = Path("/content/drive/MyDrive/MiAtlas-ML/datos-diplomado")
    else:
        CARPETA_DATOS = Path("datos-diplomado")
print("Datos:", DATOS)

## Parte 1 · k-means en 2D (30 puntos)

1. Carga los puntos y grafícalos. ¿Cuántos grupos ves? (10)
2. Aplica `KMeans` y revisa sus resultados: centroides (`cluster_centers_`), grupo de cada punto (`labels_`) y
   distancia dentro de los grupos (`inertia_`) (10).
3. Ajusta k-means con k = 1 a 10, guarda la inercia en un diccionario, grafícala y elige k con el **criterio del
   codo** (10).

In [ ]:
if DATOS == "diplomado":
    puntos = pd.read_csv(CARPETA_DATOS / "mp3_puntos.csv")
else:
    from sklearn.datasets import make_blobs
    xy, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.1, random_state=7)
    puntos = pd.DataFrame(xy, columns=["A", "B"])
sns.scatterplot(data=puntos, x="A", y="B", s=12);

In [ ]:
# ← tu código (Parte 1)

## Parte 2 · Grupos de estrellas variables (70 puntos)

1. Carga, preprocesa (¿escalar?, ¿logaritmos?), visualiza y describe los datos (20).
2. Aplica **k-means**, elige el número de grupos con el codo (y, si quieres, la silueta) e **interpreta los
   centroides**: ¿a qué tipo de estrella se parece cada grupo? (15)
3. Aplica **clustering jerárquico**, dibuja el **dendrograma** y decide cuántos grupos (10).
4. Explica en tus palabras la **mezcla de gaussianas** y aplícala con `GaussianMixture` (15).
5. Compara k-means con GMM usando gráficos y métricas (silueta; y, como aquí sí conocemos el tipo real, el índice de
   Rand ajustado). ¿Qué ventajas tiene GMM? (10)

In [ ]:
if DATOS == "diplomado":
    estrellas = pd.read_csv(CARPETA_DATOS / "mp3_estrellas.csv", index_col=0)
    tipo_real = None                                   # esta tabla no trae la clase
else:
    from datasets import load_dataset
    gcvs = load_dataset("juliensimon/gcvs-variable-stars")["train"].to_pandas()
    familias = {"EA": "eclipsante", "EB": "eclipsante", "EW": "eclipsante", "M": "período largo", "SRA": "período largo",
                "SRB": "período largo", "RRAB": "RR Lyrae", "RRC": "RR Lyrae", "DCEP": "cefeida", "DSCT": "delta Scuti",
                "HADS": "delta Scuti"}
    gcvs["clase"] = gcvs["variable_type"].str.split(":").str[0].map(familias)
    # En el GCVS, una magnitud mínima entre paréntesis («(») es directamente la amplitud
    gcvs["amplitud"] = np.where(gcvs["magnitude_min_flag"].eq("("), gcvs["magnitude_min"],
                                gcvs["magnitude_min"] - gcvs["magnitude_max"])
    gcvs = gcvs.dropna(subset=["clase", "period_days", "amplitud"])
    gcvs = gcvs[gcvs["amplitud"] > 0].sample(1000 if PRUEBA_RAPIDA else 5000, random_state=0)
    estrellas = gcvs[["period_days", "amplitud", "magnitude_max"]].reset_index(drop=True)
    tipo_real = gcvs["clase"].reset_index(drop=True)   # solo para evaluar al final: el clustering no lo ve
print(estrellas.shape)
estrellas.describe().round(2)

In [ ]:
# ← tu código (Parte 2)

*Tus conclusiones:*

**Referencia:** Jain, A. K. (2010). Data clustering: 50 years beyond K-means. *Pattern Recognition Letters*, 31(8),
651–666.

**Fuentes:** [scikit-learn · clustering (k-means, jerárquico, métricas)](https://scikit-learn.org/stable/modules/clustering.html) · [scikit-learn · mezcla de gaussianas](https://scikit-learn.org/stable/modules/mixture.html) · [SciPy · dendrograma](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html) · [GCVS en Hugging Face (CC BY 4.0)](https://huggingface.co/datasets/juliensimon/gcvs-variable-stars)